In [ ]:
from pyspark.sql import SparkSession
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['HADOOP_HOME'] = r"C:\hadoop"
os.environ['PATH'] += r"C:\hadoop\bin"
os.environ["PYSPARK_SUBMIT_ARGS"] = \
    "--jars C:/Divya/PYSPARK/insurance/jar/mysql-connector-j-8.0.33.jar pyspark-shell"

In [ ]:
spark = SparkSession.builder \
    .appName("Insurance-Medallion") \
    .master("local[*]") \
    .config("spark.jars", "C:\Divya\PYSPARK\insurance\jar\mysql-connector-j-8.0.33.jar") \
    .config("spark.driver.extraClassPath", "C:/Divya/PYSPARK/insurance/jar/mysql-connector-j-8.0.33.jar") \
    .config("spark.executor.extraClassPath", "C:/Divya/PYSPARK/insurance/jar/mysql-connector-j-8.0.33.jar") \
    .getOrCreate()

In [ ]:
claims_raw = spark.read.option("header", True).csv(r"C:\Divya\PYSPARK\insurance\input_data\raw\claims.csv")

customers_raw = spark.read.option("header", True).csv(r"C:\Divya\PYSPARK\insurance\input_data\raw\customers.csv")

policies_raw = spark.read.option("header", True).csv(r"C:\Divya\PYSPARK\insurance\input_data\raw\policies.csv")

payments_raw = spark.read.option("header", True).csv(r"C:\Divya\PYSPARK\insurance\input_data\raw\payments.csv")

In [ ]:
customers_raw.write.mode("overwrite").parquet("data/bronze/customers")
policies_raw.write.mode("overwrite").parquet("data/bronze/policies")
claims_raw.write.mode("overwrite").parquet("data/bronze/claims")
payments_raw.write.mode("overwrite").parquet("data/bronze/payments")

In [ ]:
customers = spark.read.parquet("data/bronze/customers")
policies  = spark.read.parquet("data/bronze/policies")
claims    = spark.read.parquet("data/bronze/claims")
payments  = spark.read.parquet("data/bronze/payments")

Data Cleaning + Casting

In [ ]:
from pyspark.sql.functions import col

customers_clean = customers.dropDuplicates(["customer_id"])

policies_clean = policies \
    .withColumn("premium_amount", col("premium_amount").cast("double")) \
    .dropDuplicates(["policy_id"])

claims_clean = claims \
    .withColumn("claim_amount", col("claim_amount").cast("double")) \
    .dropDuplicates(["claim_id"])

payments_clean = payments \
    .withColumn("amount", col("amount").cast("double")) \
    .dropDuplicates(["payment_id"])

Join (Core Logic)

In [ ]:
silver_df = claims_clean \
    .join(customers_clean, "customer_id", "left") \
    .join(policies_clean, "policy_id", "left") \
    .join(payments_clean, "policy_id", "left")

In [ ]:
silver_df.show()

In [ ]:

joined_df = claims.alias("cl") \
    .join(policies.alias("pl"), col("cl.policy_id") == col("pl.policy_id"), "left") \
    .join(customers.alias("cu"), col("cl.customer_id") == col("cu.customer_id"), "left") \
    .join(payments.alias("pm"), col("pl.policy_id") == col("pm.policy_id"), "left")

In [ ]:
joined_df.printSchema()

In [ ]:
silver_df = joined_df.select(
    col("cl.claim_id"),
    col("cl.policy_id"),
    col("cu.customer_id"),   # keep only ONE customer_id
    col("cl.claim_amount"),
    col("cu.city"),
    col("cu.state"),
    col("pl.policy_type"),
    col("pl.premium_amount"),
    col("pm.amount").alias("payment_amount")
)

In [ ]:
silver_df.write.mode("overwrite").parquet("data/silver/insurance")

KPI 1: Total Claim Amount by State

In [ ]:
from pyspark.sql.functions import col, sum, current_date, lit

kpi_state = silver_df.groupBy("state") \
    .agg(sum("claim_amount").alias("total_claim_amount"))

KPI 2: Total Premium by Policy Type

In [ ]:
kpi_policy = silver_df.groupBy("policy_type") \
    .agg(sum("premium_amount").alias("total_premium"))

KPI 3: High Value Claims (Fraud Detection)

In [ ]:
fraud_df = silver_df.filter(col("claim_amount") > 15000)

In [ ]:
kpi_state.write.mode("overwrite").parquet("data/gold/kpi_state")
kpi_policy.write.mode("overwrite").parquet("data/gold/kpi_policy")
fraud_df.write.mode("overwrite").parquet("data/gold/high_claims")

In [ ]:
# ==============================
# SCD TYPE 2 (DIM CUSTOMER)
# ==============================
try:
    dim_customer = spark.read.parquet("data/gold/dim_customer")
except:
    dim_customer = None

incoming = customers.select(
    "customer_id", "first_name", "last_name", "city", "state"
)

if dim_customer is None:
    # First Load
    dim_customer_new = incoming \
        .withColumn("effective_start_date", current_date()) \
        .withColumn("effective_end_date", lit(None).cast("date")) \
        .withColumn("is_current", lit("Y"))
else:
    # Join for change detection
    join_df = dim_customer.alias("old") \
        .join(incoming.alias("new"), "customer_id") \
        .filter("old.is_current = 'Y'")

    changed = join_df.filter(
        (col("old.city") != col("new.city")) |
        (col("old.state") != col("new.state"))
    )

    expired = changed.select("old.*") \
        .withColumn("effective_end_date", current_date()) \
        .withColumn("is_current", lit("N"))

    new_records = changed.select("new.*") \
        .withColumn("effective_start_date", current_date()) \
        .withColumn("effective_end_date", lit(None).cast("date")) \
        .withColumn("is_current", lit("Y"))

    unchanged = dim_customer.alias("old") \
        .join(changed.select("customer_id"), "customer_id", "left_anti")

    dim_customer_new = unchanged.unionByName(expired).unionByName(new_records)

dim_customer_new.write.mode("overwrite").parquet("data/gold/dim_customer")

# ==============================
# LOAD TO MYSQL
# ==============================
jdbc_url = "jdbc:mysql://127.0.0.1:3306/insurance_db?useSSL=false&allowPublicKeyRetrieval=true"

db_props = {
    "user": "root",
    "password": "admin@123",
    "driver": "com.mysql.cj.jdbc.Driver"
}

# 🔹 Function to write safely
def write_to_mysql(df, table_name):
    df.write \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", table_name) \
        .option("user", db_props["user"]) \
        .option("password", db_props["password"]) \
        .option("driver", db_props["driver"]) \
        .option("batchsize", 10000) \
        .option("truncate", "true") \
        .mode("overwrite") \
        .save()

# ==========================
# WRITE TABLES
# ==========================

write_to_mysql(silver_df, "insurance_silver")
write_to_mysql(kpi_policy, "kpi_policy")
write_to_mysql(kpi_state, "kpi_state")
write_to_mysql(dim_customer_new, "dim_customer")

print("✅ Pipeline executed successfully!")